# Evaluation - Optuna trial analysis

Analysis of the hyperparameter search across all pipelines: best configurations, search efficiency (pruning), optimization histories, and which choices mattered (e.g. focal vs BCE).

**Sections:** 0 Setup - 1 Best configs - 2 Search efficiency - 3 Optimization histories - 4 Loss/param analysis - 5 Summary

> Reads each pipeline's committed artifacts and reconstructs trained models via `utils.eval_protocols` (rebuilt from `best_params.json`). Run after training completes.

## 0 - Setup

In [ ]:
import sys, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_here = Path.cwd()
_nb = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from utils import eval_protocols as EP, metrics as Me, viz as V, datasets as D, explain as E, eda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = EP.ART / "evaluation" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
TAB = EP.ART / "evaluation"
PIPES = EP.available()
print("device:", device)
print("pipelines with trained results:", PIPES)
TUNED = [n for n in PIPES if EP.optuna_trials(n)]
print("pipelines with an Optuna study:", TUNED)

## 1 - Best configuration per pipeline

In [ ]:
rows = []
for n in TUNED:
    ot = EP.optuna_trials(n)
    rows.append({"pipeline": n, "best_val_auc": ot.get("best_value"), "n_trials": ot.get("n_trials"),
                 "n_complete": ot.get("n_complete"), "n_pruned": ot.get("n_pruned"), **ot.get("best_params", {})})
bp_tbl = pd.DataFrame(rows)
display(bp_tbl.round(4))
bp_tbl.to_csv(TAB / "optuna_best_params.csv", index=False)

## 2 - Search efficiency (completed vs pruned)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.45 * len(TUNED) + 2))
comp = [EP.optuna_trials(n).get("n_complete", 0) for n in TUNED]
prun = [EP.optuna_trials(n).get("n_pruned", 0) for n in TUNED]
ax.barh(TUNED, comp, color="#55A868", label="completed")
ax.barh(TUNED, prun, left=comp, color="#C44E52", label="pruned")
ax.set_xlabel("trials"); ax.set_title("Search efficiency (MedianPruner)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "optuna_search_efficiency.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 - Optimization histories (value per trial + running best)

In [ ]:
cols = 2; rows_ = math.ceil(len(TUNED) / cols)
fig, axes = plt.subplots(rows_, cols, figsize=(7 * cols, 3.0 * rows_)); axes = np.array(axes).reshape(-1)
for ax, n in zip(axes, TUNED):
    ts = EP.optuna_trials(n)["trials"]
    xs = [t["number"] for t in ts]
    ys = [t["value"] if t["value"] is not None else np.nan for t in ts]
    colors = ["#55A868" if t["state"] == "COMPLETE" else "#C44E52" for t in ts]
    ax.scatter(xs, ys, c=colors, s=18)
    run, best = [], -1.0
    for t in ts:
        if t["value"] is not None:
            best = max(best, t["value"])
        run.append(best if best >= 0 else np.nan)
    ax.plot(xs, run, "k-", lw=1, label="running best")
    ax.set_title(n, fontsize=10); ax.set_xlabel("trial"); ax.set_ylabel("val AUC"); ax.legend(fontsize=7)
for ax in axes[len(TUNED):]:
    ax.axis("off")
fig.suptitle("Optuna optimization histories (green=complete, red=pruned)", fontsize=12); fig.tight_layout()
fig.savefig(OUT / "optuna_histories.png", dpi=150, bbox_inches="tight"); plt.show()

## 4 - Did the loss function / key params matter?

Win-rate of each loss across the best configs, and (per pipeline) the saved Optuna param-importance figure is at `artifacts/<pipe>/figures/optuna_importances.png`.

In [ ]:
loss_win = pd.Series([EP.best_params(n).get("loss", "?") for n in TUNED]).value_counts()
print("loss function chosen in best config (count across pipelines):")
print(loss_win.to_string())
# completed-trial value distribution per pipeline (spread = sensitivity to HPs)
fig, ax = plt.subplots(figsize=(9, 5))
data = [[t["value"] for t in EP.optuna_trials(n)["trials"] if t["value"] is not None] for n in TUNED]
ax.boxplot(data, labels=TUNED, vert=True)
ax.set_ylabel("val AUC (completed trials)"); ax.set_title("Spread of trial scores per pipeline")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout(); fig.savefig(OUT / "optuna_value_spread.png", dpi=150, bbox_inches="tight"); plt.show()

## 5 - Summary

In [ ]:
print("Per-pipeline best val AUC and config:")
display(bp_tbl[["pipeline", "best_val_auc", "n_trials", "n_pruned"]].round(4))
print("\nTuned-search wins are the val-AUC numbers above; final test metrics are in eval-comparison.")